# 18 — Assembly101 Text-Assisted Concatenation Teacher Baselines

This notebook implements the first text-assisted Assembly101 experiment.

For every temporal position, the ground-truth action class is converted to its
stored CLIP ViT-B/16 action-text embedding:

```text
video feature:                  [512, 16]
ground-truth action text:       [512, 16]
concatenated teacher input:    [1024, 16]
```

Four controlled privileged teachers are trained:

```text
MS-TCN   + ProcedureVRL video + GT action text
MS-TCN   + CLIP video         + GT action text
LTContext + ProcedureVRL video + GT action text
LTContext + CLIP video         + GT action text
```

The temporal architecture and optimization configuration match the
corresponding visual-only baselines from notebooks 16 and 17. The only intended
change is the additional text channel.

## Critical interpretation

The text sequence is selected from **ground-truth labels**. It therefore
contains privileged information and directly encodes the correct action class.

Consequently:

- these teachers are oracle upper bounds;
- their test inference uses ground-truth text and is not deployable;
- their scores must not be compared to ordinary video-only inference as if both
  systems had the same inputs;
- the teachers are created primarily for the next knowledge-distillation stage;
- notebook 19 will train students using video only and will use text only
  indirectly through the frozen teachers during training.

## Evaluation protocol

- official Assembly101 `train / val / test`;
- model selection uses validation metrics only;
- test is evaluated once after checkpoint selection;
- the recommended teacher for distillation is selected by validation F1@25,
  not by test performance.

## 1. Mount Google Drive

In [1]:
from google.colab import drive

drive.mount(
    "/content/drive",
    force_remount=True,
)

Mounted at /content/drive


## 2. Imports

In [2]:
from pathlib import Path
from datetime import datetime, timezone

import gc
import json
import math
import random
import time
import traceback

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from tqdm.auto import tqdm

pd.set_option("display.max_columns", 160)
pd.set_option("display.max_colwidth", 220)

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print(
    "GPU:",
    torch.cuda.get_device_name(0)
    if torch.cuda.is_available()
    else "CPU",
)

PyTorch: 2.11.0+cpu
CUDA available: False
GPU: CPU


## 3. Paths and run configuration

In [3]:
DRIVE_ROOT = Path(
    "/content/drive/MyDrive/mmf_tas_lab_data"
)

FEATURE_ROOT = (
    DRIVE_ROOT
    / "text_assisted_tas"
    / "assembly101"
    / "coarse_mstcn_format"
    / "streaming_visual_features_v1"
)

PROCEDUREVRL_DATA_ROOT = (
    FEATURE_ROOT
    / "procedurevrl_hidden"
)
CLIP_DATA_ROOT = (
    FEATURE_ROOT
    / "clip_vitb16"
)

TEXT_EMBEDDING_PATH = (
    CLIP_DATA_ROOT
    / "text_embeddings"
    / "assembly101_coarse_clip_vitb16_text_embeddings.npy"
)
TEXT_METADATA_PATH = (
    CLIP_DATA_ROOT
    / "text_embeddings"
    / "assembly101_coarse_clip_vitb16_text_metadata.csv"
)

FEATURE_EXTRACTION_SUMMARY = (
    FEATURE_ROOT
    / "streaming_feature_extraction_summary.json"
)
AUTOLOOP_SUMMARY = (
    FEATURE_ROOT
    / "autoloop_latest_session_summary.json"
)

VISUAL_MSTCN_ROOT = (
    FEATURE_ROOT
    / "runs"
    / "16_visual_only_mstcn"
)
VISUAL_LTCONTEXT_ROOT = (
    FEATURE_ROOT
    / "runs"
    / "17_visual_only_ltcontext"
)

OUT_ROOT = (
    FEATURE_ROOT
    / "runs"
    / "18_text_assisted_concat_teachers"
)
OUT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

# ---------------------------------------------------------------
# Run mode
# ---------------------------------------------------------------
RUN_MODE = "full"
# RUN_MODE = "smoke"

SEED = 7

RUN_CONFIGS = {
    "smoke": {
        "epochs": 3,
        "max_train_sequences": 64,
        "max_val_sequences": 32,
        "max_test_sequences": 32,
        "eval_every": 1,
        "patience_evals": None,
        "ltcontext_warmup_epochs": 1,
    },
    "full": {
        "epochs": 120,
        "max_train_sequences": None,
        "max_val_sequences": None,
        "max_test_sequences": None,
        "eval_every": 5,
        "patience_evals": 12,
        "ltcontext_warmup_epochs": 15,
    },
}

if RUN_MODE not in RUN_CONFIGS:
    raise ValueError(
        f"Unknown RUN_MODE: {RUN_MODE}"
    )

RUN_CFG = RUN_CONFIGS[RUN_MODE]

# ---------------------------------------------------------------
# Shared data/model dimensions
# ---------------------------------------------------------------
EXPECTED_VIDEO_DIM = 512
EXPECTED_TEXT_DIM = 512
TEACHER_INPUT_DIM = (
    EXPECTED_VIDEO_DIM
    + EXPECTED_TEXT_DIM
)
EXPECTED_TEMPORAL_LENGTH = 16
EXPECTED_NUM_CLASSES = 202

BATCH_SIZE = 64
NUM_WORKERS = 0
GRAD_CLIP = 5.0

NUM_STAGES = 4

# ---------------------------------------------------------------
# MS-TCN settings copied from notebook 16
# ---------------------------------------------------------------
MSTCN_NUM_LAYERS = 6
MSTCN_NUM_F_MAPS = 64
MSTCN_DROPOUT = 0.5
MSTCN_LEARNING_RATE = 5e-4
MSTCN_WEIGHT_DECAY = 1e-4
MSTCN_SMOOTHING_WEIGHT = 0.15

# ---------------------------------------------------------------
# LTContext settings copied from notebook 17
# ---------------------------------------------------------------
LT_NUM_LAYERS = 9
LT_MODEL_DIM = 64
LT_REFINEMENT_DIM = 32
LT_NUM_HEADS = 4
LT_ATTENTION_DROPOUT = 0.1
LT_BLOCK_DROPOUT = 0.5
LT_CHANNEL_MASKING_PROB = 0.3

LT_DILATION_FACTOR = 2
LT_MAX_EFFECTIVE_DILATION = 8
LT_LOCAL_ATTENTION_RADIUS = 2
LT_LONG_TERM_ATTENTION_STRIDE = 4

LT_LEARNING_RATE = 2.5e-4
LT_WEIGHT_DECAY = 1e-3
LT_SMOOTHING_WEIGHT = 0.17

SMOOTHING_CLIP_VALUE = 16.0

STANDARDIZE_VIDEO_FEATURES = True
NORMALIZE_TEXT_EMBEDDINGS = True
TEXT_SCALE = 1.0

RESUME_IF_AVAILABLE = True
FORCE_RETRAIN_COMPLETED = False

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("RUN_MODE:", RUN_MODE)
print("RUN_CFG:", RUN_CFG)
print("DEVICE:", DEVICE)
print("Teacher input dimension:", TEACHER_INPUT_DIM)
print("TEXT_SCALE:", TEXT_SCALE)
print("OUT_ROOT:", OUT_ROOT)

RUN_MODE: full
RUN_CFG: {'epochs': 120, 'max_train_sequences': None, 'max_val_sequences': None, 'max_test_sequences': None, 'eval_every': 5, 'patience_evals': 12, 'ltcontext_warmup_epochs': 15}
DEVICE: cpu
Teacher input dimension: 1024
TEXT_SCALE: 1.0
OUT_ROOT: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/streaming_visual_features_v1/runs/18_text_assisted_concat_teachers


## 4. Verify completed feature extraction and prerequisite artifacts

In [4]:
required_paths = {
    "ProcedureVRL dataset": PROCEDUREVRL_DATA_ROOT,
    "CLIP dataset": CLIP_DATA_ROOT,
    "CLIP text embeddings": TEXT_EMBEDDING_PATH,
    "CLIP text metadata": TEXT_METADATA_PATH,
    "feature extraction summary": FEATURE_EXTRACTION_SUMMARY,
    "AUTOLOOP summary": AUTOLOOP_SUMMARY,
}

for name, path in required_paths.items():
    print(f"{name}: {path} -> {path.exists()}")

    if not path.exists():
        raise FileNotFoundError(
            f"Missing required artifact: "
            f"{name}: {path}"
        )

feature_summary = json.loads(
    FEATURE_EXTRACTION_SUMMARY.read_text(
        encoding="utf-8"
    )
)
autoloop_summary = json.loads(
    AUTOLOOP_SUMMARY.read_text(
        encoding="utf-8"
    )
)

progress = autoloop_summary.get(
    "final_progress",
    {},
)

print(
    "Feature extraction status:",
    feature_summary.get("status"),
)
print(
    "AUTOLOOP status:",
    autoloop_summary.get("status"),
)
print(
    json.dumps(
        progress,
        indent=2,
    )
)

assert progress.get(
    "complete_recordings"
) == 350
assert progress.get(
    "total_recordings"
) == 350
assert progress.get(
    "complete_sequences"
) == 680
assert progress.get(
    "total_sequences"
) == 680
assert progress.get(
    "all_complete"
) is True

print(
    "Assembly101 feature extraction: COMPLETE"
)

ProcedureVRL dataset: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/streaming_visual_features_v1/procedurevrl_hidden -> True
CLIP dataset: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/streaming_visual_features_v1/clip_vitb16 -> True
CLIP text embeddings: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/streaming_visual_features_v1/clip_vitb16/text_embeddings/assembly101_coarse_clip_vitb16_text_embeddings.npy -> True
CLIP text metadata: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/streaming_visual_features_v1/clip_vitb16/text_embeddings/assembly101_coarse_clip_vitb16_text_metadata.csv -> True
feature extraction summary: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/streaming_visual_features_v1/streaming_feature_extraction_summary.json -> True
AUTOLOOP summary: /content/dr

## 5. Reproducibility

In [5]:
def set_global_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_global_seed(SEED)

print("Seed:", SEED)

Seed: 7


## 6. Mapping and official train/validation/test split

In [6]:
def read_nonempty_lines(path):
    return [
        line.strip()
        for line in Path(path).read_text(
            encoding="utf-8",
            errors="replace",
        ).splitlines()
        if line.strip()
    ]


def load_mapping(path):
    id_to_label = {}
    label_to_id = {}

    for line in read_nonempty_lines(path):
        index_text, label = line.split(
            maxsplit=1
        )
        index = int(index_text)

        if index in id_to_label:
            raise ValueError(
                f"Duplicate class ID: {index}"
            )
        if label in label_to_id:
            raise ValueError(
                f"Duplicate class label: {label}"
            )

        id_to_label[index] = label
        label_to_id[label] = index

    return id_to_label, label_to_id


def resolve_bundle(
    split_dir,
    split_name,
):
    split_dir = Path(split_dir)

    exact_candidates = [
        split_dir
        / f"{split_name}.bundle",
        split_dir
        / f"{split_name}.split1.bundle",
    ]

    for candidate in exact_candidates:
        if candidate.exists():
            return candidate

    candidates = sorted(
        path
        for path in split_dir.glob(
            "*.bundle"
        )
        if split_name.lower()
        in path.name.lower()
        and ".partial."
        not in path.name.lower()
        and not path.name.lower().endswith(
            ".partial.bundle"
        )
    )

    if len(candidates) != 1:
        raise FileNotFoundError(
            f"Could not uniquely resolve "
            f"{split_name} in {split_dir}: "
            f"{candidates}"
        )

    return candidates[0]


def read_bundle(path):
    sequence_ids = []

    for line in read_nonempty_lines(path):
        sequence_ids.append(
            Path(line).stem
        )

    if len(sequence_ids) != len(
        set(sequence_ids)
    ):
        raise ValueError(
            f"Duplicate sequence IDs in {path}"
        )

    return sequence_ids


MAPPING_PATH = (
    PROCEDUREVRL_DATA_ROOT
    / "mapping.txt"
)
SPLIT_DIR = (
    PROCEDUREVRL_DATA_ROOT
    / "splits"
)

TRAIN_BUNDLE = resolve_bundle(
    SPLIT_DIR,
    "train",
)
VAL_BUNDLE = resolve_bundle(
    SPLIT_DIR,
    "val",
)
TEST_BUNDLE = resolve_bundle(
    SPLIT_DIR,
    "test",
)

id_to_label, label_to_id = (
    load_mapping(MAPPING_PATH)
)
num_classes = len(id_to_label)

assert num_classes == EXPECTED_NUM_CLASSES
assert set(id_to_label) == set(
    range(num_classes)
)

train_ids_full = read_bundle(
    TRAIN_BUNDLE
)
val_ids_full = read_bundle(
    VAL_BUNDLE
)
test_ids_full = read_bundle(
    TEST_BUNDLE
)

train_set = set(train_ids_full)
val_set = set(val_ids_full)
test_set = set(test_ids_full)

assert train_set.isdisjoint(val_set)
assert train_set.isdisjoint(test_set)
assert val_set.isdisjoint(test_set)

all_official_ids = (
    train_set
    | val_set
    | test_set
)

assert len(all_official_ids) == 680

print("Classes:", num_classes)
print("Train:", len(train_ids_full))
print("Validation:", len(val_ids_full))
print("Test:", len(test_ids_full))
print("Total:", len(all_official_ids))

Classes: 202
Train: 393
Validation: 120
Test: 167
Total: 680


## 7. Load and verify CLIP action-text embeddings

In [7]:
text_embeddings_raw = np.load(
    TEXT_EMBEDDING_PATH
).astype(np.float32)

text_metadata = pd.read_csv(
    TEXT_METADATA_PATH
)

required_text_columns = {
    "model_class_id",
    "action_cls",
    "prompt",
    "clip_model",
}

missing_text_columns = (
    required_text_columns
    - set(text_metadata.columns)
)

if missing_text_columns:
    raise KeyError(
        "Text metadata is missing columns: "
        f"{sorted(missing_text_columns)}"
    )

if text_embeddings_raw.shape != (
    EXPECTED_NUM_CLASSES,
    EXPECTED_TEXT_DIM,
):
    raise ValueError(
        "Unexpected text embedding shape: "
        f"{text_embeddings_raw.shape}"
    )

if text_metadata[
    "model_class_id"
].duplicated().any():
    raise ValueError(
        "Duplicate model_class_id values "
        "in text metadata."
    )

text_metadata = (
    text_metadata
    .sort_values("model_class_id")
    .reset_index(drop=True)
)

expected_class_ids = np.arange(
    EXPECTED_NUM_CLASSES
)

if not np.array_equal(
    text_metadata[
        "model_class_id"
    ].to_numpy(),
    expected_class_ids,
):
    raise ValueError(
        "Text metadata does not cover "
        "class IDs 0..201 in order."
    )

text_embeddings = (
    text_embeddings_raw[
        text_metadata.index.to_numpy()
    ]
)

alignment_rows = []

for class_id in range(num_classes):
    mapping_label = str(
        id_to_label[class_id]
    )
    metadata_label = str(
        text_metadata.loc[
            class_id,
            "action_cls",
        ]
    )

    alignment_rows.append({
        "model_class_id": class_id,
        "mapping_label": mapping_label,
        "metadata_action_cls": (
            metadata_label
        ),
        "exact_match": (
            mapping_label
            == metadata_label
        ),
        "prompt": str(
            text_metadata.loc[
                class_id,
                "prompt",
            ]
        ),
    })

alignment_df = pd.DataFrame(
    alignment_rows
)

if not alignment_df[
    "exact_match"
].all():
    display(
        alignment_df[
            ~alignment_df[
                "exact_match"
            ]
        ].head(50)
    )
    raise ValueError(
        "Text metadata action labels "
        "do not match mapping.txt."
    )

if NORMALIZE_TEXT_EMBEDDINGS:
    norms = np.linalg.norm(
        text_embeddings,
        axis=1,
        keepdims=True,
    )
    text_embeddings = (
        text_embeddings
        / np.maximum(norms, 1e-12)
    )

text_embeddings = (
    text_embeddings.astype(np.float32)
)

# Diagnostic: each stored text prototype should retrieve itself.
similarity = (
    text_embeddings
    @ text_embeddings.T
)
nearest_class = similarity.argmax(
    axis=1
)
self_retrieval_accuracy = float(
    (
        nearest_class
        == expected_class_ids
    ).mean()
)

embedding_norms = np.linalg.norm(
    text_embeddings,
    axis=1,
)

alignment_path = (
    OUT_ROOT
    / "text_mapping_alignment.csv"
)
alignment_df.to_csv(
    alignment_path,
    index=False,
)

print(
    "Text embeddings:",
    text_embeddings.shape,
)
print(
    "Text norm min/max:",
    float(embedding_norms.min()),
    float(embedding_norms.max()),
)
print(
    "Self-retrieval accuracy:",
    self_retrieval_accuracy,
)
print(
    "Prompt template example:",
    text_metadata.loc[0, "prompt"],
)
print("Saved:", alignment_path)

assert self_retrieval_accuracy == 1.0

display(alignment_df.head(20))

Text embeddings: (202, 512)
Text norm min/max: 0.9999998807907104 1.0000001192092896
Self-retrieval accuracy: 1.0
Prompt template example: a video of a person performing the action: inspect toy
Saved: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/streaming_visual_features_v1/runs/18_text_assisted_concat_teachers/text_mapping_alignment.csv


,model_class_id,mapping_label,metadata_action_cls,exact_match,prompt
0,0,inspect toy,inspect toy,True,a video of a person performing the action: inspect toy
1,1,attach cabin,attach cabin,True,a video of a person performing the action: attach cabin
2,2,detach cabin,detach cabin,True,a video of a person performing the action: detach cabin
3,3,detach wheel,detach wheel,True,a video of a person performing the action: detach wheel
4,4,attach wheel,attach wheel,True,a video of a person performing the action: attach wheel
5,5,screw chassis,screw chassis,True,a video of a person performing the action: screw chassis
6,6,demonstrate functionality,demonstrate functionality,True,a video of a person performing the action: demonstrate functionality
7,7,unscrew chassis,unscrew chassis,True,a video of a person performing the action: unscrew chassis
8,8,attach interior,attach interior,True,a video of a person performing the action: attach interior
9,9,detach roof,detach roof,True,a video of a person performing the action: detach roof


## 8. Explicit privileged-information check

In [8]:
print(
    "PRIVILEGED INPUT CHECK"
)
print(
    "At every temporal position, "
    "the teacher receives the CLIP "
    "embedding selected by the "
    "ground-truth class ID."
)
print(
    "The embedding therefore reveals "
    "the correct action identity."
)
print(
    "Teacher inference input:",
    "video + ground-truth action text",
)
print(
    "Deployable at inference:",
    False,
)
print(
    "Primary purpose:",
    "oracle teacher for later KD",
)

privileged_input_declaration = {
    "text_source": (
        "ground-truth action class "
        "at every temporal position"
    ),
    "text_embedding_model": (
        "CLIP ViT-B/16"
    ),
    "teacher_inference_input": (
        "video + ground-truth text"
    ),
    "deployable": False,
    "intended_use": (
        "oracle upper bound and "
        "knowledge-distillation teacher"
    ),
}

declaration_path = (
    OUT_ROOT
    / "privileged_input_declaration.json"
)
declaration_path.write_text(
    json.dumps(
        privileged_input_declaration,
        indent=2,
    ),
    encoding="utf-8",
)

print("Saved:", declaration_path)

PRIVILEGED INPUT CHECK
At every temporal position, the teacher receives the CLIP embedding selected by the ground-truth class ID.
The embedding therefore reveals the correct action identity.
Teacher inference input: video + ground-truth action text
Deployable at inference: False
Primary purpose: oracle teacher for later KD
Saved: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/streaming_visual_features_v1/runs/18_text_assisted_concat_teachers/privileged_input_declaration.json


## 9. Apply smoke/full sequence limits

In [9]:
def limit_ids(
    sequence_ids,
    maximum,
):
    if maximum is None:
        return list(sequence_ids)

    return list(sequence_ids)[
        : int(maximum)
    ]


train_ids = limit_ids(
    train_ids_full,
    RUN_CFG[
        "max_train_sequences"
    ],
)
val_ids = limit_ids(
    val_ids_full,
    RUN_CFG[
        "max_val_sequences"
    ],
)
test_ids = limit_ids(
    test_ids_full,
    RUN_CFG[
        "max_test_sequences"
    ],
)

print("Active train:", len(train_ids))
print("Active validation:", len(val_ids))
print("Active test:", len(test_ids))

assert train_ids
assert val_ids
assert test_ids

Active train: 393
Active validation: 120
Active test: 167


## 10. Validate both video-feature datasets

In [10]:
REPRESENTATIONS = {
    "procedurevrl_hidden": {
        "display_name": (
            "ProcedureVRL hidden"
        ),
        "data_root": (
            PROCEDUREVRL_DATA_ROOT
        ),
        "feature_description": (
            "ProcedureVRL model.head "
            "hidden video embeddings"
        ),
    },
    "clip_vitb16": {
        "display_name": (
            "CLIP ViT-B/16"
        ),
        "data_root": (
            CLIP_DATA_ROOT
        ),
        "feature_description": (
            "CLIP ViT-B/16 normalized "
            "visual embeddings"
        ),
    },
}

active_ids = (
    list(train_ids)
    + list(val_ids)
    + list(test_ids)
)

active_train_set = set(train_ids)
active_val_set = set(val_ids)

integrity_rows = []

for representation, config in (
    REPRESENTATIONS.items()
):
    data_root = Path(
        config["data_root"]
    )
    feature_dir = (
        data_root
        / "features"
    )
    gt_dir = (
        data_root
        / "groundTruth"
    )
    mapping_path = (
        data_root
        / "mapping.txt"
    )

    rep_id_to_label, _ = (
        load_mapping(mapping_path)
    )

    if rep_id_to_label != id_to_label:
        raise ValueError(
            f"Mapping differs for "
            f"{representation}"
        )

    for sequence_id in tqdm(
        active_ids,
        desc=(
            f"Validate {representation}"
        ),
    ):
        feature_path = (
            feature_dir
            / f"{sequence_id}.npy"
        )
        gt_path = (
            gt_dir
            / f"{sequence_id}.txt"
        )

        if not feature_path.exists():
            raise FileNotFoundError(
                feature_path
            )
        if not gt_path.exists():
            raise FileNotFoundError(
                gt_path
            )

        feature = np.load(
            feature_path,
            mmap_mode="r",
        )
        labels = read_nonempty_lines(
            gt_path
        )

        unknown_labels = sorted(
            set(labels)
            - set(label_to_id)
        )

        split_name = (
            "train"
            if sequence_id
            in active_train_set
            else (
                "val"
                if sequence_id
                in active_val_set
                else "test"
            )
        )

        integrity_rows.append({
            "representation": (
                representation
            ),
            "sequence_id": sequence_id,
            "split": split_name,
            "feature_shape": str(
                tuple(feature.shape)
            ),
            "gt_length": len(labels),
            "finite": bool(
                np.isfinite(feature).all()
            ),
            "unknown_label_count": (
                len(unknown_labels)
            ),
        })

        if feature.shape != (
            EXPECTED_VIDEO_DIM,
            EXPECTED_TEMPORAL_LENGTH,
        ):
            raise ValueError(
                f"Unexpected shape for "
                f"{representation}/"
                f"{sequence_id}: "
                f"{feature.shape}"
            )

        if len(labels) != (
            EXPECTED_TEMPORAL_LENGTH
        ):
            raise ValueError(
                f"Unexpected GT length for "
                f"{sequence_id}: "
                f"{len(labels)}"
            )

        if not np.isfinite(
            feature
        ).all():
            raise ValueError(
                f"NaN/Inf in {feature_path}"
            )

        if unknown_labels:
            raise ValueError(
                f"Unknown labels in "
                f"{gt_path}: "
                f"{unknown_labels}"
            )

integrity_df = pd.DataFrame(
    integrity_rows
)

integrity_path = (
    OUT_ROOT
    / f"dataset_integrity_{RUN_MODE}.csv"
)
integrity_df.to_csv(
    integrity_path,
    index=False,
)

display(
    integrity_df.groupby(
        [
            "representation",
            "split",
        ]
    ).agg(
        sequences=(
            "sequence_id",
            "size",
        ),
        feature_shapes=(
            "feature_shape",
            "nunique",
        ),
        gt_lengths=(
            "gt_length",
            "nunique",
        ),
        finite=(
            "finite",
            "all",
        ),
        unknown_labels=(
            "unknown_label_count",
            "sum",
        ),
    ).reset_index()
)

print("Saved:", integrity_path)

Validate procedurevrl_hidden:   0%|          | 0/680 [00:00<?, ?it/s]

Validate clip_vitb16:   0%|          | 0/680 [00:00<?, ?it/s]

,representation,split,sequences,feature_shapes,gt_lengths,finite,unknown_labels
0,clip_vitb16,test,167,1,1,True,0
1,clip_vitb16,train,393,1,1,True,0
2,clip_vitb16,val,120,1,1,True,0
3,procedurevrl_hidden,test,167,1,1,True,0
4,procedurevrl_hidden,train,393,1,1,True,0
5,procedurevrl_hidden,val,120,1,1,True,0


Saved: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/streaming_visual_features_v1/runs/18_text_assisted_concat_teachers/dataset_integrity_full.csv


## 11. Train-only video normalization and privileged dataset

In [11]:
def compute_train_stats(
    feature_dir,
    sequence_ids,
):
    feature_dir = Path(
        feature_dir
    )

    total_sum = None
    total_sumsq = None
    total_count = 0

    for sequence_id in tqdm(
        sequence_ids,
        desc="Train video statistics",
    ):
        feature = np.load(
            feature_dir
            / f"{sequence_id}.npy"
        ).astype(np.float64)

        if total_sum is None:
            total_sum = np.zeros(
                feature.shape[0],
                dtype=np.float64,
            )
            total_sumsq = np.zeros(
                feature.shape[0],
                dtype=np.float64,
            )

        total_sum += feature.sum(
            axis=1
        )
        total_sumsq += (
            feature ** 2
        ).sum(axis=1)
        total_count += (
            feature.shape[1]
        )

    mean = total_sum / total_count
    variance = (
        total_sumsq
        / total_count
        - mean ** 2
    )
    variance = np.maximum(
        variance,
        1e-12,
    )
    std = np.sqrt(variance)

    return (
        mean.astype(np.float32),
        std.astype(np.float32),
    )


class Assembly101PrivilegedTextDataset(
    Dataset
):
    def __init__(
        self,
        sequence_ids,
        data_root,
        label_to_id,
        text_embeddings,
        video_mean,
        video_std,
    ):
        self.sequence_ids = list(
            sequence_ids
        )
        self.data_root = Path(
            data_root
        )
        self.feature_dir = (
            self.data_root
            / "features"
        )
        self.gt_dir = (
            self.data_root
            / "groundTruth"
        )
        self.label_to_id = dict(
            label_to_id
        )
        self.text_embeddings = (
            np.asarray(
                text_embeddings,
                dtype=np.float32,
            )
        )
        self.video_mean = (
            np.asarray(
                video_mean,
                dtype=np.float32,
            )
        )
        self.video_std = (
            np.asarray(
                video_std,
                dtype=np.float32,
            )
        )

    def __len__(self):
        return len(
            self.sequence_ids
        )

    def __getitem__(self, index):
        sequence_id = (
            self.sequence_ids[index]
        )

        video = np.load(
            self.feature_dir
            / f"{sequence_id}.npy"
        ).astype(np.float32)

        label_names = (
            read_nonempty_lines(
                self.gt_dir
                / f"{sequence_id}.txt"
            )
        )

        labels = np.asarray(
            [
                self.label_to_id[
                    label_name
                ]
                for label_name
                in label_names
            ],
            dtype=np.int64,
        )

        if STANDARDIZE_VIDEO_FEATURES:
            video = (
                video
                - self.video_mean[
                    :, None
                ]
            ) / (
                self.video_std[
                    :, None
                ]
                + 1e-8
            )

        # Ground-truth class ID selects the privileged text prototype.
        privileged_text = (
            self.text_embeddings[
                labels
            ]
            .T
            .astype(np.float32)
        )

        privileged_text *= float(
            TEXT_SCALE
        )

        return {
            "sequence_id": (
                sequence_id
            ),
            "video": torch.from_numpy(
                video
            ),
            "privileged_text": (
                torch.from_numpy(
                    privileged_text
                )
            ),
            "labels": torch.from_numpy(
                labels
            ),
        }


print(
    "Privileged dataset class ready."
)

Privileged dataset class ready.


## 12. Temporal action segmentation metrics

In [12]:
CANDIDATE_IGNORE_CLASSES = {
    "background",
    "SIL",
    "silence",
}

ignore_class_ids = {
    label_to_id[label]
    for label
    in CANDIDATE_IGNORE_CLASSES
    if label in label_to_id
}

print(
    "Ignored class IDs:",
    ignore_class_ids,
)


def collapse_segments(
    frame_labels,
    ignored_ids=None,
):
    ignored_ids = set(
        ignored_ids or []
    )

    labels = []
    starts = []
    ends = []

    active_label = None

    for index, label in enumerate(
        frame_labels
    ):
        label = int(label)

        if label in ignored_ids:
            if active_label is not None:
                ends.append(index)
                active_label = None
            continue

        if label != active_label:
            if active_label is not None:
                ends.append(index)

            labels.append(label)
            starts.append(index)
            active_label = label

    if active_label is not None:
        ends.append(
            len(frame_labels)
        )

    return labels, starts, ends


def levenshtein_distance(
    predicted,
    target,
):
    rows = len(predicted) + 1
    columns = len(target) + 1

    distance = np.zeros(
        (rows, columns),
        dtype=np.int32,
    )

    distance[:, 0] = np.arange(
        rows
    )
    distance[0, :] = np.arange(
        columns
    )

    for row in range(
        1,
        rows,
    ):
        for column in range(
            1,
            columns,
        ):
            substitution_cost = (
                0
                if predicted[row - 1]
                == target[column - 1]
                else 1
            )

            distance[
                row,
                column,
            ] = min(
                distance[
                    row - 1,
                    column,
                ] + 1,
                distance[
                    row,
                    column - 1,
                ] + 1,
                distance[
                    row - 1,
                    column - 1,
                ] + substitution_cost,
            )

    return int(
        distance[-1, -1]
    )


def edit_score_single(
    prediction,
    target,
    ignored_ids=None,
):
    predicted_segments, _, _ = (
        collapse_segments(
            prediction,
            ignored_ids,
        )
    )
    target_segments, _, _ = (
        collapse_segments(
            target,
            ignored_ids,
        )
    )

    if (
        not predicted_segments
        and not target_segments
    ):
        return 100.0

    denominator = max(
        len(predicted_segments),
        len(target_segments),
    )

    if denominator == 0:
        return 0.0

    distance = (
        levenshtein_distance(
            predicted_segments,
            target_segments,
        )
    )

    return (
        1.0
        - distance / denominator
    ) * 100.0


def f_score_single(
    prediction,
    target,
    overlap,
    ignored_ids=None,
):
    (
        predicted_labels,
        predicted_starts,
        predicted_ends,
    ) = collapse_segments(
        prediction,
        ignored_ids,
    )

    (
        target_labels,
        target_starts,
        target_ends,
    ) = collapse_segments(
        target,
        ignored_ids,
    )

    true_positives = 0
    false_positives = 0

    hits = np.zeros(
        len(target_labels),
        dtype=np.float32,
    )

    for predicted_index in range(
        len(predicted_labels)
    ):
        best_iou = 0.0
        best_target_index = -1

        for target_index in range(
            len(target_labels)
        ):
            if (
                predicted_labels[
                    predicted_index
                ]
                != target_labels[
                    target_index
                ]
            ):
                continue

            intersection = (
                min(
                    predicted_ends[
                        predicted_index
                    ],
                    target_ends[
                        target_index
                    ],
                )
                - max(
                    predicted_starts[
                        predicted_index
                    ],
                    target_starts[
                        target_index
                    ],
                )
            )

            union = (
                max(
                    predicted_ends[
                        predicted_index
                    ],
                    target_ends[
                        target_index
                    ],
                )
                - min(
                    predicted_starts[
                        predicted_index
                    ],
                    target_starts[
                        target_index
                    ],
                )
            )

            iou = (
                max(
                    intersection,
                    0,
                )
                / union
                if union > 0
                else 0.0
            )

            if iou > best_iou:
                best_iou = iou
                best_target_index = (
                    target_index
                )

        if (
            best_iou >= overlap
            and best_target_index >= 0
            and hits[
                best_target_index
            ] == 0
        ):
            true_positives += 1
            hits[
                best_target_index
            ] = 1
        else:
            false_positives += 1

    false_negatives = (
        len(target_labels)
        - int(hits.sum())
    )

    return (
        true_positives,
        false_positives,
        false_negatives,
    )


def compute_metrics(
    predictions,
    targets,
    ignored_ids=None,
):
    total_correct = 0
    total_positions = 0
    edit_scores = []

    f_statistics = {
        0.10: [0, 0, 0],
        0.25: [0, 0, 0],
        0.50: [0, 0, 0],
    }

    for sequence_id, prediction in (
        predictions.items()
    ):
        target = targets[
            sequence_id
        ]

        if len(prediction) != len(
            target
        ):
            raise ValueError(
                f"Length mismatch for "
                f"{sequence_id}"
            )

        total_correct += int(
            (
                prediction
                == target
            ).sum()
        )
        total_positions += len(
            target
        )

        edit_scores.append(
            edit_score_single(
                prediction,
                target,
                ignored_ids,
            )
        )

        for overlap in f_statistics:
            tp, fp, fn = f_score_single(
                prediction,
                target,
                overlap,
                ignored_ids,
            )

            f_statistics[
                overlap
            ][0] += tp
            f_statistics[
                overlap
            ][1] += fp
            f_statistics[
                overlap
            ][2] += fn

    metrics = {
        "acc": (
            100.0
            * total_correct
            / max(
                total_positions,
                1,
            )
        ),
        "edit": (
            float(
                np.mean(
                    edit_scores
                )
            )
            if edit_scores
            else 0.0
        ),
    }

    for overlap, (
        tp,
        fp,
        fn,
    ) in f_statistics.items():
        precision = (
            tp
            / max(
                tp + fp,
                1e-8,
            )
        )
        recall = (
            tp
            / max(
                tp + fn,
                1e-8,
            )
        )

        f1 = (
            2.0
            * precision
            * recall
            / max(
                precision + recall,
                1e-8,
            )
        )

        metrics[
            f"f1@{int(overlap * 100)}"
        ] = 100.0 * f1

    return metrics


def rounded_metrics(metrics):
    return {
        key: round(
            float(value),
            2,
        )
        for key, value
        in metrics.items()
    }


print("Metric implementation ready.")

Ignored class IDs: set()
Metric implementation ready.


## 13. Shared multi-stage loss

In [13]:
def multistage_segmentation_loss(
    outputs,
    targets,
    smoothing_weight,
):
    # outputs:
    # [stages, batch, classes, time]
    # targets:
    # [batch, time]

    total_loss = torch.zeros(
        (),
        device=outputs.device,
    )

    cross_entropy_total = 0.0
    smoothing_total = 0.0

    for stage_index in range(
        outputs.shape[0]
    ):
        logits = outputs[
            stage_index
        ]

        cross_entropy = F.cross_entropy(
            logits.permute(
                0,
                2,
                1,
            ).reshape(
                -1,
                logits.shape[1],
            ),
            targets.reshape(-1),
        )

        log_probabilities = (
            F.log_softmax(
                logits,
                dim=1,
            )
        )

        smoothing = F.mse_loss(
            log_probabilities[
                :,
                :,
                1:,
            ],
            log_probabilities.detach()[
                :,
                :,
                :-1,
            ],
            reduction="none",
        )

        smoothing = torch.clamp(
            smoothing,
            min=0.0,
            max=SMOOTHING_CLIP_VALUE,
        ).mean()

        total_loss = (
            total_loss
            + cross_entropy
            + smoothing_weight
            * smoothing
        )

        cross_entropy_total += float(
            cross_entropy.detach()
            .cpu()
            .item()
        )
        smoothing_total += float(
            smoothing.detach()
            .cpu()
            .item()
        )

    parts = {
        "cross_entropy": (
            cross_entropy_total
        ),
        "smoothing": (
            smoothing_total
        ),
        "total": float(
            total_loss.detach()
            .cpu()
            .item()
        ),
    }

    return total_loss, parts


print("Loss ready.")

Loss ready.


## 14. MS-TCN teacher architecture

In [14]:
class MSTCNDilatedResidualLayer(
    nn.Module
):
    def __init__(
        self,
        dilation,
        channels,
        dropout,
    ):
        super().__init__()

        self.conv_dilated = nn.Conv1d(
            channels,
            channels,
            kernel_size=3,
            padding=dilation,
            dilation=dilation,
        )
        self.conv_1x1 = nn.Conv1d(
            channels,
            channels,
            kernel_size=1,
        )
        self.dropout = nn.Dropout(
            dropout
        )

    def forward(self, inputs):
        output = F.relu(
            self.conv_dilated(
                inputs
            )
        )
        output = self.conv_1x1(
            output
        )
        output = self.dropout(
            output
        )

        return inputs + output


class MSTCNSingleStage(nn.Module):
    def __init__(
        self,
        input_dim,
        num_classes,
    ):
        super().__init__()

        self.input_projection = (
            nn.Conv1d(
                input_dim,
                MSTCN_NUM_F_MAPS,
                kernel_size=1,
            )
        )

        self.layers = nn.ModuleList([
            MSTCNDilatedResidualLayer(
                dilation=(
                    2 ** layer_index
                ),
                channels=(
                    MSTCN_NUM_F_MAPS
                ),
                dropout=MSTCN_DROPOUT,
            )
            for layer_index in range(
                MSTCN_NUM_LAYERS
            )
        ])

        self.output_projection = (
            nn.Conv1d(
                MSTCN_NUM_F_MAPS,
                num_classes,
                kernel_size=1,
            )
        )

    def forward(self, inputs):
        output = self.input_projection(
            inputs
        )

        for layer in self.layers:
            output = layer(output)

        return self.output_projection(
            output
        )


class MSTCNTeacher(nn.Module):
    def __init__(
        self,
        num_classes,
    ):
        super().__init__()

        self.stage1 = (
            MSTCNSingleStage(
                input_dim=(
                    TEACHER_INPUT_DIM
                ),
                num_classes=num_classes,
            )
        )

        self.refinement_stages = (
            nn.ModuleList([
                MSTCNSingleStage(
                    input_dim=num_classes,
                    num_classes=(
                        num_classes
                    ),
                )
                for _ in range(
                    NUM_STAGES - 1
                )
            ])
        )

    def forward(
        self,
        video,
        privileged_text,
    ):
        teacher_input = torch.cat(
            [
                video,
                privileged_text,
            ],
            dim=1,
        )

        logits = self.stage1(
            teacher_input
        )
        outputs = [logits]

        for stage in (
            self.refinement_stages
        ):
            logits = stage(
                F.softmax(
                    logits,
                    dim=1,
                )
            )
            outputs.append(logits)

        return torch.stack(
            outputs,
            dim=0,
        )


print("MS-TCN teacher ready.")

MS-TCN teacher ready.


## 15. LTContext teacher architecture

In [15]:
class LocalWindowAttention(nn.Module):
    def __init__(
        self,
        model_dim,
        num_heads,
        radius,
        dropout,
    ):
        super().__init__()

        self.radius = int(radius)
        self.attention = (
            nn.MultiheadAttention(
                embed_dim=model_dim,
                num_heads=num_heads,
                dropout=dropout,
                batch_first=True,
            )
        )

    def forward(
        self,
        query_key,
        value_context=None,
    ):
        query = query_key.transpose(
            1,
            2,
        )
        value = (
            query
            if value_context is None
            else value_context.transpose(
                1,
                2,
            )
        )

        time_steps = query.shape[1]

        positions = torch.arange(
            time_steps,
            device=query.device,
        )

        distance = torch.abs(
            positions[:, None]
            - positions[None, :]
        )

        attention_mask = (
            distance
            > self.radius
        )

        output, _ = self.attention(
            query=query,
            key=query,
            value=value,
            attn_mask=attention_mask,
            need_weights=False,
        )

        return output.transpose(
            1,
            2,
        )


class StridedLongTermAttention(
    nn.Module
):
    def __init__(
        self,
        model_dim,
        num_heads,
        stride,
        dropout,
    ):
        super().__init__()

        self.stride = int(stride)
        self.attention = (
            nn.MultiheadAttention(
                embed_dim=model_dim,
                num_heads=num_heads,
                dropout=dropout,
                batch_first=True,
            )
        )

    def forward(
        self,
        query_key,
        value_context=None,
    ):
        query_all = query_key.transpose(
            1,
            2,
        )
        value_all = (
            query_all
            if value_context is None
            else value_context.transpose(
                1,
                2,
            )
        )

        output_all = torch.zeros_like(
            query_all
        )

        time_steps = (
            query_all.shape[1]
        )

        for offset in range(
            min(
                self.stride,
                time_steps,
            )
        ):
            indices = torch.arange(
                offset,
                time_steps,
                self.stride,
                device=query_all.device,
            )

            if indices.numel() == 0:
                continue

            query = query_all.index_select(
                1,
                indices,
            )
            value = value_all.index_select(
                1,
                indices,
            )

            output, _ = self.attention(
                query=query,
                key=query,
                value=value,
                need_weights=False,
            )

            output_all.index_copy_(
                1,
                indices,
                output,
            )

        return output_all.transpose(
            1,
            2,
        )


class LTContextBlock(nn.Module):
    def __init__(
        self,
        model_dim,
        dilation,
    ):
        super().__init__()

        self.dilated_conv = nn.Conv1d(
            model_dim,
            model_dim,
            kernel_size=3,
            padding=dilation,
            dilation=dilation,
        )

        self.local_norm = (
            nn.InstanceNorm1d(
                model_dim,
                affine=True,
            )
        )
        self.local_attention = (
            LocalWindowAttention(
                model_dim=model_dim,
                num_heads=LT_NUM_HEADS,
                radius=(
                    LT_LOCAL_ATTENTION_RADIUS
                ),
                dropout=(
                    LT_ATTENTION_DROPOUT
                ),
            )
        )

        self.long_norm = (
            nn.InstanceNorm1d(
                model_dim,
                affine=True,
            )
        )
        self.long_attention = (
            StridedLongTermAttention(
                model_dim=model_dim,
                num_heads=LT_NUM_HEADS,
                stride=(
                    LT_LONG_TERM_ATTENTION_STRIDE
                ),
                dropout=(
                    LT_ATTENTION_DROPOUT
                ),
            )
        )

        self.output_projection = (
            nn.Conv1d(
                model_dim,
                model_dim,
                kernel_size=1,
            )
        )
        self.dropout = nn.Dropout(
            LT_BLOCK_DROPOUT
        )

    def forward(
        self,
        inputs,
        previous_stage_features=None,
    ):
        output = F.gelu(
            self.dilated_conv(
                inputs
            )
        )

        output = (
            output
            + self.local_attention(
                self.local_norm(output),
                previous_stage_features,
            )
        )

        output = (
            output
            + self.long_attention(
                self.long_norm(output),
                previous_stage_features,
            )
        )

        output = self.output_projection(
            output
        )
        output = self.dropout(
            output
        )

        return inputs + output


class LTContextStage(nn.Module):
    def __init__(
        self,
        input_dim,
        model_dim,
        num_classes,
    ):
        super().__init__()

        self.channel_dropout = (
            nn.Dropout1d(
                LT_CHANNEL_MASKING_PROB
            )
        )

        self.input_projection = (
            nn.Conv1d(
                input_dim,
                model_dim,
                kernel_size=1,
            )
        )

        dilations = [
            min(
                (
                    LT_DILATION_FACTOR
                    ** layer_index
                ),
                LT_MAX_EFFECTIVE_DILATION,
            )
            for layer_index in range(
                LT_NUM_LAYERS
            )
        ]

        self.blocks = nn.ModuleList([
            LTContextBlock(
                model_dim=model_dim,
                dilation=dilation,
            )
            for dilation in dilations
        ])

        self.output_projection = (
            nn.Conv1d(
                model_dim,
                num_classes,
                kernel_size=1,
            )
        )

    def forward(
        self,
        inputs,
        previous_stage_features=None,
    ):
        features = self.input_projection(
            self.channel_dropout(
                inputs
            )
        )

        for block in self.blocks:
            features = block(
                features,
                previous_stage_features,
            )

        logits = self.output_projection(
            features
        )

        return logits, features


class LTContextTeacher(nn.Module):
    def __init__(
        self,
        num_classes,
    ):
        super().__init__()

        self.stage1 = LTContextStage(
            input_dim=(
                TEACHER_INPUT_DIM
            ),
            model_dim=(
                LT_MODEL_DIM
            ),
            num_classes=num_classes,
        )

        self.dimension_reduction = (
            nn.Conv1d(
                LT_MODEL_DIM,
                LT_REFINEMENT_DIM,
                kernel_size=1,
            )
        )

        self.refinement_stages = (
            nn.ModuleList([
                LTContextStage(
                    input_dim=(
                        num_classes
                    ),
                    model_dim=(
                        LT_REFINEMENT_DIM
                    ),
                    num_classes=(
                        num_classes
                    ),
                )
                for _ in range(
                    NUM_STAGES - 1
                )
            ])
        )

    def forward(
        self,
        video,
        privileged_text,
    ):
        teacher_input = torch.cat(
            [
                video,
                privileged_text,
            ],
            dim=1,
        )

        logits, features = (
            self.stage1(
                teacher_input
            )
        )
        outputs = [logits]

        previous_features = (
            self.dimension_reduction(
                features
            )
        )

        for stage in (
            self.refinement_stages
        ):
            logits, features = stage(
                F.softmax(
                    logits,
                    dim=1,
                ),
                previous_stage_features=(
                    previous_features
                ),
            )

            outputs.append(logits)
            previous_features = (
                features
            )

        return torch.stack(
            outputs,
            dim=0,
        )


print("LTContext teacher ready.")

LTContext teacher ready.


## 16. Model factories and forward-shape tests

In [16]:
MODEL_CONFIGS = {
    "mstcn": {
        "display_name": "MS-TCN",
        "learning_rate": (
            MSTCN_LEARNING_RATE
        ),
        "weight_decay": (
            MSTCN_WEIGHT_DECAY
        ),
        "smoothing_weight": (
            MSTCN_SMOOTHING_WEIGHT
        ),
        "warmup_epochs": 0,
    },
    "ltcontext": {
        "display_name": "LTContext",
        "learning_rate": (
            LT_LEARNING_RATE
        ),
        "weight_decay": (
            LT_WEIGHT_DECAY
        ),
        "smoothing_weight": (
            LT_SMOOTHING_WEIGHT
        ),
        "warmup_epochs": (
            RUN_CFG[
                "ltcontext_warmup_epochs"
            ]
        ),
    },
}


def create_teacher(
    temporal_model,
):
    if temporal_model == "mstcn":
        return MSTCNTeacher(
            num_classes=num_classes
        ).to(DEVICE)

    if temporal_model == "ltcontext":
        return LTContextTeacher(
            num_classes=num_classes
        ).to(DEVICE)

    raise ValueError(
        f"Unknown temporal model: "
        f"{temporal_model}"
    )


parameter_counts = {}

for temporal_model in MODEL_CONFIGS:
    test_model = create_teacher(
        temporal_model
    )

    parameter_counts[
        temporal_model
    ] = sum(
        parameter.numel()
        for parameter
        in test_model.parameters()
    )

    with torch.no_grad():
        test_video = torch.zeros(
            2,
            EXPECTED_VIDEO_DIM,
            EXPECTED_TEMPORAL_LENGTH,
            device=DEVICE,
        )
        test_text = torch.zeros(
            2,
            EXPECTED_TEXT_DIM,
            EXPECTED_TEMPORAL_LENGTH,
            device=DEVICE,
        )
        test_output = test_model(
            test_video,
            test_text,
        )

    print(
        temporal_model,
        "parameters:",
        parameter_counts[
            temporal_model
        ],
    )
    print(
        temporal_model,
        "output shape:",
        tuple(
            test_output.shape
        ),
    )

    assert test_output.shape == (
        NUM_STAGES,
        2,
        num_classes,
        EXPECTED_TEMPORAL_LENGTH,
    )

    del (
        test_model,
        test_video,
        test_text,
        test_output,
    )
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

mstcn parameters: 553384
mstcn output shape: (4, 2, 202, 16)
ltcontext parameters: 914600
ltcontext output shape: (4, 2, 202, 16)


## 17. Teacher evaluation

In [17]:
@torch.no_grad()
def evaluate_teacher(
    model,
    loader,
    save_predictions=False,
    prediction_dir=None,
):
    model.eval()

    predictions = {}
    targets = {}

    if save_predictions:
        prediction_dir = Path(
            prediction_dir
        )
        prediction_dir.mkdir(
            parents=True,
            exist_ok=True,
        )

    for batch in loader:
        video = batch[
            "video"
        ].to(
            DEVICE,
            non_blocking=True,
        )
        privileged_text = batch[
            "privileged_text"
        ].to(
            DEVICE,
            non_blocking=True,
        )
        labels = batch[
            "labels"
        ].cpu().numpy()
        sequence_ids = list(
            batch["sequence_id"]
        )

        outputs = model(
            video,
            privileged_text,
        )

        prediction_batch = (
            outputs[-1]
            .argmax(dim=1)
            .detach()
            .cpu()
            .numpy()
        )

        for batch_index, sequence_id in enumerate(
            sequence_ids
        ):
            prediction = (
                prediction_batch[
                    batch_index
                ].astype(np.int64)
            )
            target = labels[
                batch_index
            ].astype(np.int64)

            predictions[
                sequence_id
            ] = prediction
            targets[
                sequence_id
            ] = target

            if save_predictions:
                prediction_labels = [
                    id_to_label[
                        int(class_id)
                    ]
                    for class_id
                    in prediction
                ]

                (
                    prediction_dir
                    / f"{sequence_id}.txt"
                ).write_text(
                    "\n".join(
                        prediction_labels
                    )
                    + "\n",
                    encoding="utf-8",
                )

    metrics = compute_metrics(
        predictions,
        targets,
        ignored_ids=(
            ignore_class_ids
        ),
    )

    return (
        metrics,
        predictions,
        targets,
    )


def torch_load_checkpoint(path):
    try:
        return torch.load(
            path,
            map_location=DEVICE,
            weights_only=False,
        )
    except TypeError:
        return torch.load(
            path,
            map_location=DEVICE,
        )


print("Teacher evaluation ready.")

Teacher evaluation ready.


## 18. Load matching visual-only baseline summaries

In [18]:
def visual_summary_path(
    temporal_model,
    representation,
):
    if temporal_model == "mstcn":
        root = VISUAL_MSTCN_ROOT
    elif temporal_model == "ltcontext":
        root = VISUAL_LTCONTEXT_ROOT
    else:
        raise ValueError(
            temporal_model
        )

    return (
        root
        / representation
        / RUN_MODE
        / "results"
        / "final_summary.json"
    )


visual_baseline_summaries = {}

for temporal_model in MODEL_CONFIGS:
    for representation in (
        REPRESENTATIONS
    ):
        key = (
            temporal_model,
            representation,
        )

        path = visual_summary_path(
            temporal_model,
            representation,
        )

        print(
            key,
            "->",
            path,
            "exists=",
            path.exists(),
        )

        if path.exists():
            summary = json.loads(
                path.read_text(
                    encoding="utf-8"
                )
            )

            if summary.get(
                "status"
            ) != "completed":
                raise RuntimeError(
                    f"Visual baseline is not "
                    f"completed: {path}"
                )

            visual_baseline_summaries[
                key
            ] = summary

if RUN_MODE == "full":
    assert len(
        visual_baseline_summaries
    ) == 4

print(
    "Loaded visual baselines:",
    len(
        visual_baseline_summaries
    ),
)

('mstcn', 'procedurevrl_hidden') -> /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/streaming_visual_features_v1/runs/16_visual_only_mstcn/procedurevrl_hidden/full/results/final_summary.json exists= True
('mstcn', 'clip_vitb16') -> /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/streaming_visual_features_v1/runs/16_visual_only_mstcn/clip_vitb16/full/results/final_summary.json exists= True
('ltcontext', 'procedurevrl_hidden') -> /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/streaming_visual_features_v1/runs/17_visual_only_ltcontext/procedurevrl_hidden/full/results/final_summary.json exists= True
('ltcontext', 'clip_vitb16') -> /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/streaming_visual_features_v1/runs/17_visual_only_ltcontext/clip_vitb16/full/results/final_summary.json exists= True
Loaded visual baselines: 4


## 19. Controlled privileged-teacher training

In [19]:
def create_scheduler(
    optimizer,
    temporal_model,
):
    total_epochs = int(
        RUN_CFG["epochs"]
    )

    warmup_epochs = int(
        MODEL_CONFIGS[
            temporal_model
        ]["warmup_epochs"]
    )

    if warmup_epochs <= 0:
        return (
            torch.optim.lr_scheduler
            .CosineAnnealingLR(
                optimizer,
                T_max=total_epochs,
            )
        )

    def learning_rate_factor(
        epoch_index,
    ):
        current_epoch = (
            epoch_index + 1
        )

        if current_epoch <= (
            warmup_epochs
        ):
            return (
                current_epoch
                / warmup_epochs
            )

        cosine_length = max(
            total_epochs
            - warmup_epochs,
            1,
        )

        cosine_position = min(
            max(
                current_epoch
                - warmup_epochs,
                0,
            ),
            cosine_length,
        )

        return 0.5 * (
            1.0
            + math.cos(
                math.pi
                * cosine_position
                / cosine_length
            )
        )

    return (
        torch.optim.lr_scheduler
        .LambdaLR(
            optimizer,
            lr_lambda=(
                learning_rate_factor
            ),
        )
    )


def run_teacher_experiment(
    temporal_model,
    representation,
):
    model_config = (
        MODEL_CONFIGS[
            temporal_model
        ]
    )
    representation_config = (
        REPRESENTATIONS[
            representation
        ]
    )

    data_root = Path(
        representation_config[
            "data_root"
        ]
    )

    experiment_root = (
        OUT_ROOT
        / temporal_model
        / representation
        / RUN_MODE
    )
    model_dir = (
        experiment_root
        / "models"
    )
    prediction_dir = (
        experiment_root
        / "predictions"
    )
    result_dir = (
        experiment_root
        / "results"
    )

    for directory in [
        experiment_root,
        model_dir,
        prediction_dir,
        result_dir,
    ]:
        directory.mkdir(
            parents=True,
            exist_ok=True,
        )

    final_summary_path = (
        result_dir
        / "final_summary.json"
    )

    if (
        final_summary_path.exists()
        and not FORCE_RETRAIN_COMPLETED
    ):
        existing_summary = json.loads(
            final_summary_path.read_text(
                encoding="utf-8"
            )
        )

        if existing_summary.get(
            "status"
        ) == "completed":
            print(
                "\nSkipping completed teacher:",
                temporal_model,
                representation,
            )

            return existing_summary

    feature_dir = (
        data_root
        / "features"
    )

    if STANDARDIZE_VIDEO_FEATURES:
        train_mean, train_std = (
            compute_train_stats(
                feature_dir,
                train_ids,
            )
        )
    else:
        train_mean = np.zeros(
            EXPECTED_VIDEO_DIM,
            dtype=np.float32,
        )
        train_std = np.ones(
            EXPECTED_VIDEO_DIM,
            dtype=np.float32,
        )

    mean_path = (
        result_dir
        / "train_video_mean.npy"
    )
    std_path = (
        result_dir
        / "train_video_std.npy"
    )

    np.save(
        mean_path,
        train_mean,
    )
    np.save(
        std_path,
        train_std,
    )

    train_dataset = (
        Assembly101PrivilegedTextDataset(
            sequence_ids=train_ids,
            data_root=data_root,
            label_to_id=label_to_id,
            text_embeddings=(
                text_embeddings
            ),
            video_mean=train_mean,
            video_std=train_std,
        )
    )
    val_dataset = (
        Assembly101PrivilegedTextDataset(
            sequence_ids=val_ids,
            data_root=data_root,
            label_to_id=label_to_id,
            text_embeddings=(
                text_embeddings
            ),
            video_mean=train_mean,
            video_std=train_std,
        )
    )
    test_dataset = (
        Assembly101PrivilegedTextDataset(
            sequence_ids=test_ids,
            data_root=data_root,
            label_to_id=label_to_id,
            text_embeddings=(
                text_embeddings
            ),
            video_mean=train_mean,
            video_std=train_std,
        )
    )

    generator = torch.Generator()
    generator.manual_seed(SEED)

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=(
            torch.cuda.is_available()
        ),
        generator=generator,
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=(
            torch.cuda.is_available()
        ),
    )
    test_loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=(
            torch.cuda.is_available()
        ),
    )

    sample_batch = next(
        iter(train_loader)
    )

    assert sample_batch[
        "video"
    ].shape[1:] == (
        EXPECTED_VIDEO_DIM,
        EXPECTED_TEMPORAL_LENGTH,
    )
    assert sample_batch[
        "privileged_text"
    ].shape[1:] == (
        EXPECTED_TEXT_DIM,
        EXPECTED_TEMPORAL_LENGTH,
    )
    assert sample_batch[
        "labels"
    ].shape[1:] == (
        EXPECTED_TEMPORAL_LENGTH,
    )

    print("\n" + "=" * 88)
    print(
        "PRIVILEGED TEACHER:",
        temporal_model,
        "+",
        representation,
    )
    print("=" * 88)
    print(
        "Train/val/test:",
        len(train_dataset),
        len(val_dataset),
        len(test_dataset),
    )
    print(
        "Video batch:",
        tuple(
            sample_batch[
                "video"
            ].shape
        ),
    )
    print(
        "Text batch:",
        tuple(
            sample_batch[
                "privileged_text"
            ].shape
        ),
    )
    print(
        "Concatenated input dimension:",
        TEACHER_INPUT_DIM,
    )
    print(
        "Deployable:",
        False,
    )

    set_global_seed(SEED)

    model = create_teacher(
        temporal_model
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=model_config[
            "learning_rate"
        ],
        weight_decay=model_config[
            "weight_decay"
        ],
    )

    scheduler = create_scheduler(
        optimizer,
        temporal_model,
    )

    best_model_path = (
        model_dir
        / "best_teacher.pt"
    )
    last_checkpoint_path = (
        model_dir
        / "last_checkpoint.pt"
    )
    history_path = (
        result_dir
        / "training_history.csv"
    )

    start_epoch = 1
    best_score = -math.inf
    best_epoch = None
    best_val_metrics = None
    no_improvement_evals = 0
    history = []

    if (
        RESUME_IF_AVAILABLE
        and last_checkpoint_path.exists()
    ):
        checkpoint = (
            torch_load_checkpoint(
                last_checkpoint_path
            )
        )

        if (
            checkpoint.get(
                "temporal_model"
            )
            == temporal_model
            and checkpoint.get(
                "representation"
            )
            == representation
            and checkpoint.get(
                "run_mode"
            )
            == RUN_MODE
        ):
            model.load_state_dict(
                checkpoint[
                    "model_state_dict"
                ]
            )
            optimizer.load_state_dict(
                checkpoint[
                    "optimizer_state_dict"
                ]
            )
            scheduler.load_state_dict(
                checkpoint[
                    "scheduler_state_dict"
                ]
            )

            start_epoch = (
                int(
                    checkpoint[
                        "epoch"
                    ]
                )
                + 1
            )
            best_score = float(
                checkpoint.get(
                    "best_score",
                    -math.inf,
                )
            )
            best_epoch = (
                checkpoint.get(
                    "best_epoch"
                )
            )
            best_val_metrics = (
                checkpoint.get(
                    "best_val_metrics"
                )
            )
            no_improvement_evals = int(
                checkpoint.get(
                    "no_improvement_evals",
                    0,
                )
            )
            history = list(
                checkpoint.get(
                    "history",
                    [],
                )
            )

            print(
                "Resuming from epoch:",
                start_epoch,
            )

    training_started = time.time()
    stopped_early = False

    for epoch in range(
        start_epoch,
        RUN_CFG["epochs"] + 1,
    ):
        model.train()

        epoch_losses = []
        epoch_cross_entropy = []
        epoch_smoothing = []

        for batch in train_loader:
            video = batch[
                "video"
            ].to(
                DEVICE,
                non_blocking=True,
            )
            privileged_text = batch[
                "privileged_text"
            ].to(
                DEVICE,
                non_blocking=True,
            )
            labels = batch[
                "labels"
            ].to(
                DEVICE,
                non_blocking=True,
            )

            optimizer.zero_grad(
                set_to_none=True
            )

            outputs = model(
                video,
                privileged_text,
            )

            loss, loss_parts = (
                multistage_segmentation_loss(
                    outputs=outputs,
                    targets=labels,
                    smoothing_weight=(
                        model_config[
                            "smoothing_weight"
                        ]
                    ),
                )
            )

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                GRAD_CLIP,
            )

            optimizer.step()

            epoch_losses.append(
                loss_parts["total"]
            )
            epoch_cross_entropy.append(
                loss_parts[
                    "cross_entropy"
                ]
            )
            epoch_smoothing.append(
                loss_parts[
                    "smoothing"
                ]
            )

        scheduler.step()

        history_row = {
            "epoch": epoch,
            "train_loss": float(
                np.mean(
                    epoch_losses
                )
            ),
            "train_cross_entropy": float(
                np.mean(
                    epoch_cross_entropy
                )
            ),
            "train_smoothing": float(
                np.mean(
                    epoch_smoothing
                )
            ),
            "learning_rate": float(
                scheduler.get_last_lr()[0]
            ),
        }

        evaluate_now = (
            epoch == 1
            or epoch
            % RUN_CFG[
                "eval_every"
            ]
            == 0
            or epoch
            == RUN_CFG["epochs"]
        )

        if evaluate_now:
            (
                val_metrics,
                _,
                _,
            ) = evaluate_teacher(
                model,
                val_loader,
                save_predictions=False,
            )

            history_row.update({
                f"val_{key}": value
                for key, value
                in val_metrics.items()
            })

            selection_score = (
                val_metrics["f1@25"]
                + 0.01
                * val_metrics["edit"]
            )

            improved = (
                selection_score
                > best_score
            )

            if improved:
                best_score = (
                    selection_score
                )
                best_epoch = epoch
                best_val_metrics = dict(
                    val_metrics
                )
                no_improvement_evals = 0

                torch.save(
                    {
                        "epoch": epoch,
                        "run_mode": RUN_MODE,
                        "temporal_model": (
                            temporal_model
                        ),
                        "representation": (
                            representation
                        ),
                        "teacher_type": (
                            "privileged_concat"
                        ),
                        "deployable": False,
                        "model_state_dict": (
                            model.state_dict()
                        ),
                        "validation_metrics": (
                            val_metrics
                        ),
                        "selection_score": (
                            selection_score
                        ),
                        "input_config": {
                            "video_dim": (
                                EXPECTED_VIDEO_DIM
                            ),
                            "text_dim": (
                                EXPECTED_TEXT_DIM
                            ),
                            "concat_dim": (
                                TEACHER_INPUT_DIM
                            ),
                            "temporal_length": (
                                EXPECTED_TEMPORAL_LENGTH
                            ),
                            "text_source": (
                                "ground-truth "
                                "action classes"
                            ),
                            "text_scale": (
                                TEXT_SCALE
                            ),
                        },
                        "model_config": (
                            model_config
                        ),
                    },
                    best_model_path,
                )
            else:
                no_improvement_evals += 1

            print(
                f"{temporal_model}/"
                f"{representation} "
                f"epoch {epoch:03d} "
                f"loss="
                f"{history_row['train_loss']:.4f} "
                f"val_acc="
                f"{val_metrics['acc']:.2f} "
                f"val_edit="
                f"{val_metrics['edit']:.2f} "
                f"val_f1@10="
                f"{val_metrics['f1@10']:.2f} "
                f"val_f1@25="
                f"{val_metrics['f1@25']:.2f} "
                f"val_f1@50="
                f"{val_metrics['f1@50']:.2f} "
                f"best="
                f"{'yes' if improved else 'no'}"
            )
        else:
            print(
                f"{temporal_model}/"
                f"{representation} "
                f"epoch {epoch:03d} "
                f"loss="
                f"{history_row['train_loss']:.4f}"
            )

        history.append(
            history_row
        )

        torch.save(
            {
                "epoch": epoch,
                "run_mode": RUN_MODE,
                "temporal_model": (
                    temporal_model
                ),
                "representation": (
                    representation
                ),
                "teacher_type": (
                    "privileged_concat"
                ),
                "model_state_dict": (
                    model.state_dict()
                ),
                "optimizer_state_dict": (
                    optimizer.state_dict()
                ),
                "scheduler_state_dict": (
                    scheduler.state_dict()
                ),
                "best_score": best_score,
                "best_epoch": best_epoch,
                "best_val_metrics": (
                    best_val_metrics
                ),
                "no_improvement_evals": (
                    no_improvement_evals
                ),
                "history": history,
            },
            last_checkpoint_path,
        )

        pd.DataFrame(
            history
        ).to_csv(
            history_path,
            index=False,
        )

        patience = RUN_CFG[
            "patience_evals"
        ]

        if (
            evaluate_now
            and patience is not None
            and no_improvement_evals
            >= patience
        ):
            stopped_early = True

            print(
                "Early stopping after",
                no_improvement_evals,
                "validation evaluations "
                "without improvement.",
            )
            break

    training_minutes = (
        time.time()
        - training_started
    ) / 60.0

    if not best_model_path.exists():
        raise RuntimeError(
            "No best teacher checkpoint "
            f"was saved for "
            f"{temporal_model}/"
            f"{representation}"
        )

    best_checkpoint = (
        torch_load_checkpoint(
            best_model_path
        )
    )

    model.load_state_dict(
        best_checkpoint[
            "model_state_dict"
        ]
    )
    model.eval()

    (
        final_val_metrics,
        _,
        _,
    ) = evaluate_teacher(
        model,
        val_loader,
        save_predictions=False,
    )

    test_prediction_dir = (
        prediction_dir
        / "best_teacher_test"
    )

    (
        test_metrics,
        _,
        _,
    ) = evaluate_teacher(
        model,
        test_loader,
        save_predictions=True,
        prediction_dir=(
            test_prediction_dir
        ),
    )

    prediction_count = len(
        list(
            test_prediction_dir.glob(
                "*.txt"
            )
        )
    )

    if prediction_count != len(
        test_ids
    ):
        raise RuntimeError(
            "Unexpected number of "
            "test prediction files: "
            f"{prediction_count}"
        )

    visual_summary = (
        visual_baseline_summaries.get(
            (
                temporal_model,
                representation,
            )
        )
    )
    visual_metrics = (
        visual_summary.get(
            "test_metrics"
        )
        if visual_summary
        else None
    )

    result_row = {
        "experiment": (
            "assembly101_"
            f"{temporal_model}_"
            f"{representation}_"
            "privileged_concat_teacher"
        ),
        "temporal_model": (
            model_config[
                "display_name"
            ]
        ),
        "representation": (
            representation
        ),
        "video_feature_shape": (
            "[512, 16]"
        ),
        "text_feature_shape": (
            "[512, 16]"
        ),
        "teacher_input_shape": (
            "[1024, 16]"
        ),
        "training_input": (
            "video + ground-truth "
            "action text"
        ),
        "inference_input": (
            "video + ground-truth "
            "action text"
        ),
        "deployable": False,
        "best_epoch": int(
            best_checkpoint["epoch"]
        ),
        "val_acc": round(
            final_val_metrics["acc"],
            2,
        ),
        "val_edit": round(
            final_val_metrics["edit"],
            2,
        ),
        "val_f1@10": round(
            final_val_metrics[
                "f1@10"
            ],
            2,
        ),
        "val_f1@25": round(
            final_val_metrics[
                "f1@25"
            ],
            2,
        ),
        "val_f1@50": round(
            final_val_metrics[
                "f1@50"
            ],
            2,
        ),
        "test_acc": round(
            test_metrics["acc"],
            2,
        ),
        "test_edit": round(
            test_metrics["edit"],
            2,
        ),
        "test_f1@10": round(
            test_metrics["f1@10"],
            2,
        ),
        "test_f1@25": round(
            test_metrics["f1@25"],
            2,
        ),
        "test_f1@50": round(
            test_metrics["f1@50"],
            2,
        ),
    }

    if visual_metrics:
        for metric_name in [
            "acc",
            "edit",
            "f1@10",
            "f1@25",
            "f1@50",
        ]:
            result_row[
                f"delta_vs_visual_"
                f"{metric_name}"
            ] = round(
                float(
                    test_metrics[
                        metric_name
                    ]
                )
                - float(
                    visual_metrics[
                        metric_name
                    ]
                ),
                2,
            )

    result_df = pd.DataFrame([
        result_row
    ])

    result_csv_path = (
        result_dir
        / "final_metrics.csv"
    )
    result_md_path = (
        result_dir
        / "final_metrics.md"
    )

    result_df.to_csv(
        result_csv_path,
        index=False,
    )
    result_md_path.write_text(
        result_df.to_markdown(
            index=False
        ),
        encoding="utf-8",
    )

    summary = {
        "status": "completed",
        "run_mode": RUN_MODE,
        "experiment": (
            result_row[
                "experiment"
            ]
        ),
        "teacher_type": (
            "privileged ground-truth "
            "text concatenation"
        ),
        "deployable": False,
        "temporal_model": (
            temporal_model
        ),
        "representation": (
            representation
        ),
        "input": {
            "video_shape": [
                EXPECTED_VIDEO_DIM,
                EXPECTED_TEMPORAL_LENGTH,
            ],
            "text_shape": [
                EXPECTED_TEXT_DIM,
                EXPECTED_TEMPORAL_LENGTH,
            ],
            "concatenated_shape": [
                TEACHER_INPUT_DIM,
                EXPECTED_TEMPORAL_LENGTH,
            ],
            "text_source": (
                "ground-truth action "
                "class at each temporal "
                "position"
            ),
            "text_embedding_path": str(
                TEXT_EMBEDDING_PATH
            ),
            "text_metadata_path": str(
                TEXT_METADATA_PATH
            ),
            "text_scale": TEXT_SCALE,
        },
        "data": {
            "num_classes": num_classes,
            "num_train_sequences": (
                len(train_ids)
            ),
            "num_val_sequences": (
                len(val_ids)
            ),
            "num_test_sequences": (
                len(test_ids)
            ),
        },
        "selection_protocol": (
            "best checkpoint selected by "
            "validation F1@25 + "
            "0.01 * validation Edit; "
            "test evaluated once"
        ),
        "model": {
            "parameter_count": sum(
                parameter.numel()
                for parameter
                in model.parameters()
            ),
            "config": model_config,
        },
        "training": {
            "maximum_epochs": (
                RUN_CFG["epochs"]
            ),
            "best_epoch": int(
                best_checkpoint["epoch"]
            ),
            "stopped_early": (
                stopped_early
            ),
            "training_minutes_this_run": (
                training_minutes
            ),
            "batch_size": BATCH_SIZE,
            "seed": SEED,
        },
        "validation_metrics": (
            rounded_metrics(
                final_val_metrics
            )
        ),
        "test_metrics": (
            rounded_metrics(
                test_metrics
            )
        ),
        "matching_visual_baseline": (
            visual_summary
        ),
        "paths": {
            "best_teacher": str(
                best_model_path
            ),
            "last_checkpoint": str(
                last_checkpoint_path
            ),
            "training_history": str(
                history_path
            ),
            "test_predictions": str(
                test_prediction_dir
            ),
            "final_metrics_csv": str(
                result_csv_path
            ),
            "train_video_mean": str(
                mean_path
            ),
            "train_video_std": str(
                std_path
            ),
        },
        "interpretation": (
            "Oracle teacher. "
            "Ground-truth text is required "
            "at inference and directly "
            "encodes the target class. "
            "Use this checkpoint as a "
            "teacher for video-only KD."
        ),
    }

    final_summary_path.write_text(
        json.dumps(
            summary,
            indent=2,
        ),
        encoding="utf-8",
    )

    display(result_df)
    print(
        json.dumps(
            {
                "status": (
                    summary["status"]
                ),
                "experiment": (
                    summary[
                        "experiment"
                    ]
                ),
                "validation_metrics": (
                    summary[
                        "validation_metrics"
                    ]
                ),
                "test_metrics": (
                    summary[
                        "test_metrics"
                    ]
                ),
                "best_teacher": (
                    summary[
                        "paths"
                    ][
                        "best_teacher"
                    ]
                ),
            },
            indent=2,
        )
    )

    del (
        model,
        optimizer,
        scheduler,
        train_loader,
        val_loader,
        test_loader,
        train_dataset,
        val_dataset,
        test_dataset,
        sample_batch,
    )

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return summary

## 20. Train all four concatenation teachers

In [20]:
teacher_summaries = {}

for temporal_model in [
    "mstcn",
    "ltcontext",
]:
    for representation in (
        REPRESENTATIONS
    ):
        key = (
            temporal_model,
            representation,
        )

        try:
            teacher_summaries[
                key
            ] = run_teacher_experiment(
                temporal_model=(
                    temporal_model
                ),
                representation=(
                    representation
                ),
            )
        except Exception:
            error_dir = (
                OUT_ROOT
                / "errors"
            )
            error_dir.mkdir(
                parents=True,
                exist_ok=True,
            )

            error_path = (
                error_dir
                / (
                    f"{temporal_model}_"
                    f"{representation}_"
                    "traceback.txt"
                )
            )

            error_path.write_text(
                traceback.format_exc(),
                encoding="utf-8",
            )

            print(
                "Saved traceback:",
                error_path,
            )
            raise

print("\nCompleted teachers:")

for key, summary in (
    teacher_summaries.items()
):
    print(
        key,
        "->",
        summary["status"],
        summary["test_metrics"],
    )


Skipping completed teacher: mstcn procedurevrl_hidden

Skipping completed teacher: mstcn clip_vitb16

Skipping completed teacher: ltcontext procedurevrl_hidden

Skipping completed teacher: ltcontext clip_vitb16

Completed teachers:
('mstcn', 'procedurevrl_hidden') -> completed {'acc': 41.39, 'edit': 32.7, 'f1@10': 37.41, 'f1@25': 35.96, 'f1@50': 31.93}
('mstcn', 'clip_vitb16') -> completed {'acc': 44.27, 'edit': 37.39, 'f1@10': 41.65, 'f1@25': 40.94, 'f1@50': 36.81}
('ltcontext', 'procedurevrl_hidden') -> completed {'acc': 35.25, 'edit': 26.4, 'f1@10': 31.61, 'f1@25': 26.8, 'f1@50': 19.27}
('ltcontext', 'clip_vitb16') -> completed {'acc': 34.51, 'edit': 27.34, 'f1@10': 31.37, 'f1@25': 26.98, 'f1@50': 20.38}


## 21. Final comparisons and teacher recommendation for KD

In [21]:
comparison_rows = []

for (
    temporal_model,
    representation,
), summary in teacher_summaries.items():
    validation_metrics = (
        summary[
            "validation_metrics"
        ]
    )
    test_metrics = (
        summary[
            "test_metrics"
        ]
    )

    visual_summary = (
        visual_baseline_summaries.get(
            (
                temporal_model,
                representation,
            )
        )
    )
    visual_metrics = (
        visual_summary[
            "test_metrics"
        ]
        if visual_summary
        else {}
    )

    row = {
        "temporal_model": (
            MODEL_CONFIGS[
                temporal_model
            ]["display_name"]
        ),
        "temporal_model_key": (
            temporal_model
        ),
        "representation": (
            representation
        ),
        "teacher_input": (
            "video + GT text"
        ),
        "deployable": False,
        "best_epoch": (
            summary[
                "training"
            ]["best_epoch"]
        ),
        "val_acc": (
            validation_metrics[
                "acc"
            ]
        ),
        "val_edit": (
            validation_metrics[
                "edit"
            ]
        ),
        "val_f1@10": (
            validation_metrics[
                "f1@10"
            ]
        ),
        "val_f1@25": (
            validation_metrics[
                "f1@25"
            ]
        ),
        "val_f1@50": (
            validation_metrics[
                "f1@50"
            ]
        ),
        "test_acc": (
            test_metrics["acc"]
        ),
        "test_edit": (
            test_metrics["edit"]
        ),
        "test_f1@10": (
            test_metrics["f1@10"]
        ),
        "test_f1@25": (
            test_metrics["f1@25"]
        ),
        "test_f1@50": (
            test_metrics["f1@50"]
        ),
        "visual_test_acc": (
            visual_metrics.get("acc")
        ),
        "visual_test_edit": (
            visual_metrics.get("edit")
        ),
        "visual_test_f1@10": (
            visual_metrics.get(
                "f1@10"
            )
        ),
        "visual_test_f1@25": (
            visual_metrics.get(
                "f1@25"
            )
        ),
        "visual_test_f1@50": (
            visual_metrics.get(
                "f1@50"
            )
        ),
        "best_teacher_path": (
            summary[
                "paths"
            ][
                "best_teacher"
            ]
        ),
    }

    for metric_name in [
        "acc",
        "edit",
        "f1@10",
        "f1@25",
        "f1@50",
    ]:
        if metric_name in (
            visual_metrics
        ):
            row[
                f"teacher_minus_visual_"
                f"{metric_name}"
            ] = round(
                float(
                    test_metrics[
                        metric_name
                    ]
                )
                - float(
                    visual_metrics[
                        metric_name
                    ]
                ),
                2,
            )

    comparison_rows.append(row)

comparison_df = pd.DataFrame(
    comparison_rows
)

# Teacher recommendation is based only on validation data.
comparison_df[
    "validation_selection_score"
] = (
    comparison_df["val_f1@25"]
    + 0.01
    * comparison_df["val_edit"]
)

comparison_df = (
    comparison_df
    .sort_values(
        [
            "validation_selection_score",
            "val_f1@25",
            "val_edit",
        ],
        ascending=False,
    )
    .reset_index(drop=True)
)

recommended_row = (
    comparison_df.iloc[0]
)

comparison_csv_path = (
    OUT_ROOT
    / (
        "concat_teacher_"
        f"comparison_{RUN_MODE}.csv"
    )
)
comparison_md_path = (
    OUT_ROOT
    / (
        "concat_teacher_"
        f"comparison_{RUN_MODE}.md"
    )
)

comparison_df.to_csv(
    comparison_csv_path,
    index=False,
)
comparison_md_path.write_text(
    comparison_df.to_markdown(
        index=False
    ),
    encoding="utf-8",
)

display(comparison_df)

recommended_teacher = {
    "selection_basis": (
        "validation F1@25 + "
        "0.01 * validation Edit"
    ),
    "temporal_model": str(
        recommended_row[
            "temporal_model_key"
        ]
    ),
    "representation": str(
        recommended_row[
            "representation"
        ]
    ),
    "validation_selection_score": (
        float(
            recommended_row[
                "validation_selection_score"
            ]
        )
    ),
    "validation_metrics": {
        "acc": float(
            recommended_row[
                "val_acc"
            ]
        ),
        "edit": float(
            recommended_row[
                "val_edit"
            ]
        ),
        "f1@10": float(
            recommended_row[
                "val_f1@10"
            ]
        ),
        "f1@25": float(
            recommended_row[
                "val_f1@25"
            ]
        ),
        "f1@50": float(
            recommended_row[
                "val_f1@50"
            ]
        ),
    },
    "teacher_checkpoint": str(
        recommended_row[
            "best_teacher_path"
        ]
    ),
    "deployable": False,
}

final_summary = {
    "status": "completed",
    "run_mode": RUN_MODE,
    "task": (
        "Assembly101 privileged "
        "ground-truth-text "
        "concatenation teachers"
    ),
    "privileged_input": (
        privileged_input_declaration
    ),
    "experiments": {
        (
            f"{temporal_model}/"
            f"{representation}"
        ): summary
        for (
            temporal_model,
            representation,
        ), summary
        in teacher_summaries.items()
    },
    "recommended_teacher_for_kd": (
        recommended_teacher
    ),
    "selection_warning": (
        "The teacher recommendation "
        "uses validation metrics only. "
        "Teacher test scores are oracle "
        "scores because test ground-truth "
        "text is provided."
    ),
    "paths": {
        "comparison_csv": str(
            comparison_csv_path
        ),
        "comparison_markdown": str(
            comparison_md_path
        ),
        "output_root": str(
            OUT_ROOT
        ),
    },
    "next_step": (
        "Train controlled CE-only and "
        "KD video-only students. "
        "Student inference must use "
        "video features only."
    ),
    "created_utc": (
        datetime.now(
            timezone.utc
        ).isoformat()
    ),
}

final_summary_path = (
    OUT_ROOT
    / f"final_summary_{RUN_MODE}.json"
)

final_summary_path.write_text(
    json.dumps(
        final_summary,
        indent=2,
    ),
    encoding="utf-8",
)

print(
    "Recommended teacher for KD:"
)
print(
    json.dumps(
        recommended_teacher,
        indent=2,
    )
)
print(
    "Saved:",
    comparison_csv_path,
)
print(
    "Saved:",
    comparison_md_path,
)
print(
    "Saved:",
    final_summary_path,
)
print("\nNext notebook:")
print(
    "19_assembly101_video_only_"
    "knowledge_distillation_"
    "students_COLAB.ipynb"
)

,temporal_model,temporal_model_key,representation,teacher_input,deployable,best_epoch,val_acc,val_edit,val_f1@10,val_f1@25,val_f1@50,test_acc,test_edit,test_f1@10,test_f1@25,test_f1@50,visual_test_acc,visual_test_edit,visual_test_f1@10,visual_test_f1@25,visual_test_f1@50,best_teacher_path,teacher_minus_visual_acc,teacher_minus_visual_edit,teacher_minus_visual_f1@10,teacher_minus_visual_f1@25,teacher_minus_visual_f1@50,validation_selection_score
0,MS-TCN,mstcn,clip_vitb16,video + GT text,False,120,45.21,34.92,39.73,38.25,34.14,44.27,37.39,41.65,40.94,36.81,25.86,25.01,28.00,23.82,17.58,/content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/streaming_visual_features_v1/runs/18_text_assisted_concat_teachers/mstcn/clip_vitb16/full/models/best_teacher.pt,18.41,12.38,13.65,17.12,19.23,38.5992
1,MS-TCN,mstcn,procedurevrl_hidden,video + GT text,False,105,42.86,33.61,37.20,36.01,32.24,41.39,32.70,37.41,35.96,31.93,26.16,24.74,28.36,23.79,15.72,/content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/streaming_visual_features_v1/runs/18_text_assisted_concat_teachers/mstcn/procedurevrl_hidden/full/models/best_teacher.pt,15.23,7.96,9.05,12.17,16.21,36.3461
2,LTContext,ltcontext,procedurevrl_hidden,video + GT text,False,105,38.33,29.70,33.93,29.99,22.11,35.25,26.40,31.61,26.80,19.27,27.10,22.80,25.11,20.05,11.92,/content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/streaming_visual_features_v1/runs/18_text_assisted_concat_teachers/ltcontext/procedurevrl_hidden/full/models/best_teacher.pt,8.15,3.60,6.50,6.75,7.35,30.2870
3,LTContext,ltcontext,clip_vitb16,video + GT text,False,95,37.86,28.61,32.81,29.78,21.20,34.51,27.34,31.37,26.98,20.38,27.10,22.70,25.62,21.96,13.39,/content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/streaming_visual_features_v1/runs/18_text_assisted_concat_teachers/ltcontext/clip_vitb16/full/models/best_teacher.pt,7.41,4.64,5.75,5.02,6.99,30.0661


Recommended teacher for KD:
{
  "selection_basis": "validation F1@25 + 0.01 * validation Edit",
  "temporal_model": "mstcn",
  "representation": "clip_vitb16",
  "validation_selection_score": 38.5992,
  "validation_metrics": {
    "acc": 45.21,
    "edit": 34.92,
    "f1@10": 39.73,
    "f1@25": 38.25,
    "f1@50": 34.14
  },
  "teacher_checkpoint": "/content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/streaming_visual_features_v1/runs/18_text_assisted_concat_teachers/mstcn/clip_vitb16/full/models/best_teacher.pt",
  "deployable": false
}
Saved: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/streaming_visual_features_v1/runs/18_text_assisted_concat_teachers/concat_teacher_comparison_full.csv
Saved: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/streaming_visual_features_v1/runs/18_text_assisted_concat_teachers/concat_teacher_comparison_full.md
Saved: /content/drive

## Completion criterion

A successful full run completes four oracle teachers:

```text
MS-TCN   / ProcedureVRL
MS-TCN   / CLIP
LTContext / ProcedureVRL
LTContext / CLIP
```

and writes:

```text
.../streaming_visual_features_v1/runs/18_text_assisted_concat_teachers/
├── mstcn/
│   ├── procedurevrl_hidden/full/
│   └── clip_vitb16/full/
├── ltcontext/
│   ├── procedurevrl_hidden/full/
│   └── clip_vitb16/full/
├── concat_teacher_comparison_full.csv
├── concat_teacher_comparison_full.md
├── final_summary_full.json
├── privileged_input_declaration.json
└── text_mapping_alignment.csv
```

The important output for notebook 19 is each `best_teacher.pt` checkpoint.
Notebook 19 will train video-only students and will compare:

```text
controlled CE-only student
vs
video-only KD student
```

under the same architecture, representation, split, seed, and optimization
configuration.